# Blender纯点云渲染器 - Google Colab版本

本notebook用于在Google Colab上使用GPU渲染纯点云（所有粒子使用相同颜色）

## 使用步骤：
1. 确保运行时类型设置为GPU（运行时 -> 更改运行时类型 -> GPU）
2. 运行所有单元格（会自动从GitHub克隆代码）
3. 上传PLY文件到Colab
4. 配置渲染参数（包括粒子颜色）
5. 执行渲染


## 1. 从GitHub克隆仓库并安装依赖


In [ ]:
# 从GitHub克隆仓库
import os

print("正在从GitHub克隆仓库...")
!git clone https://github.com/EvaShenLu/PointCloud_Render.git

# 切换到仓库目录
%cd PointCloud_Render

# 安装Python依赖
print("\n正在安装Python依赖...")
%pip install -q numpy plyfile

print("\n仓库克隆和依赖安装完成！")
print(f"工作目录: {os.getcwd()}")


## 2. 安装Blender并设置环境


In [ ]:
# 安装Blender（headless版本，适用于服务器环境）
import subprocess
import sys

print("正在安装Blender...")

# 下载Blender（使用3.6 LTS版本，稳定且支持GPU）
BLENDER_VERSION = "3.6.5"
BLENDER_URL = f"https://download.blender.org/release/Blender3.6/blender-{BLENDER_VERSION}-linux-x64.tar.xz"

os.makedirs("/content/blender", exist_ok=True)

# 下载Blender
!wget -q {BLENDER_URL} -O /content/blender.tar.xz
!tar -xf /content/blender.tar.xz -C /content/blender --strip-components=1
!rm /content/blender.tar.xz

# 安装Python依赖到Blender的Python环境
BLENDER_PYTHON = "/content/blender/3.6/python/bin/python3.10"
!{BLENDER_PYTHON} -m ensurepip
!{BLENDER_PYTHON} -m pip install -q numpy

print("Blender安装完成！")
print(f"Blender路径: /content/blender/blender")
print(f"Python路径: {BLENDER_PYTHON}")

# 创建必要的目录
os.makedirs("/content/input_ply", exist_ok=True)
os.makedirs("/content/output_images", exist_ok=True)

print("\n目录创建完成！")
print("请使用左侧文件浏览器上传PLY文件到 /content/input_ply/ 目录")


In [ ]:
# 验证仓库文件是否存在
import os

REPO_DIR = "/content/PointCloud_Render"
SCRIPT_PATH = os.path.join(REPO_DIR, "blender_point_cloud_renderer.py")

if os.path.exists(SCRIPT_PATH):
    print(f"✓ 找到渲染脚本: {SCRIPT_PATH}")
    print(f"✓ 仓库目录: {REPO_DIR}")
else:
    print(f"✗ 未找到渲染脚本: {SCRIPT_PATH}")
    print("请检查仓库是否成功克隆")


## 3. 配置渲染参数


In [ ]:
# ========== 渲染配置 ==========
# 修改这些参数来调整渲染设置

# 输入PLY文件路径（相对于/content/input_ply/）
PLY_FILE = "frame_0299.ply"  # 修改为你的PLY文件名

# 输出文件夹
OUTPUT_FOLDER = "/content/output_images"

# 图像尺寸
IMAGE_WIDTH = 1920
IMAGE_HEIGHT = 1080

# 渲染引擎 ('cycles' 或 'eevee')
RENDER_ENGINE = "cycles"  # cycles质量更好但更慢，eevee更快但质量略低

# Cycles采样数（仅cycles引擎有效）
SAMPLES = 64  # 增加采样数可提升质量但会变慢（建议32-128）

# 批次渲染选项（可选）
USE_BATCH_RENDERING = False  # 设为True启用批次渲染
MAX_BATCHES = None  # 例如：10 表示只渲染前10个批次（约20,480个粒子）
SINGLE_BATCH_ID = None  # 例如：0 表示只渲染批次0（约2,048个粒子）

# LOD降采样（可选）
MAX_POINTS = None  # 例如：50000 表示最多渲染50,000个点（None表示不限制）

# ==============================

print("渲染配置：")
print(f"  PLY文件: {PLY_FILE}")
print(f"  输出文件夹: {OUTPUT_FOLDER}")
print(f"  图像尺寸: {IMAGE_WIDTH}x{IMAGE_HEIGHT}")
print(f"  渲染引擎: {RENDER_ENGINE}")
print(f"  采样数: {SAMPLES}")
print(f"  粒子颜色: 灰色 (0.3, 0.3, 0.3) - 默认值")
if MAX_POINTS is not None:
    print(f"  最大点数: {MAX_POINTS:,} (LOD降采样)")
if USE_BATCH_RENDERING:
    if SINGLE_BATCH_ID is not None:
        print(f"  批次渲染: 单个批次 {SINGLE_BATCH_ID}")
    elif MAX_BATCHES is not None:
        print(f"  批次渲染: 前 {MAX_BATCHES} 个批次")
    else:
        print(f"  批次渲染: 所有批次")
else:
    print(f"  批次渲染: 禁用（渲染所有粒子）")


## 4. 执行渲染


In [ ]:
# 检查PLY文件是否存在
ply_path = f"/content/input_ply/{PLY_FILE}"
REPO_DIR = "/content/PointCloud_Render"
SCRIPT_PATH = os.path.join(REPO_DIR, "blender_point_cloud_renderer.py")

if not os.path.exists(ply_path):
    print(f"错误: PLY文件不存在: {ply_path}")
    print("\n请确保：")
    print("1. 已上传PLY文件到 /content/input_ply/ 目录")
    print("2. PLY_FILE变量中的文件名正确")
elif not os.path.exists(SCRIPT_PATH):
    print(f"错误: 渲染脚本不存在: {SCRIPT_PATH}")
    print("请检查仓库是否成功克隆")
else:
    print(f"✓ 找到PLY文件: {ply_path}")
    print(f"✓ 找到渲染脚本: {SCRIPT_PATH}")
    
    # 创建Python脚本用于Blender执行
    render_script = f'''
import sys
sys.path.insert(0, '{REPO_DIR}')

from blender_point_cloud_renderer import BlenderPointCloudRenderer

# 创建渲染器（使用默认灰色 (0.3, 0.3, 0.3)）
renderer = BlenderPointCloudRenderer(
    file_path="{ply_path}",
    output_folder="{OUTPUT_FOLDER}",
    image_width={IMAGE_WIDTH},
    image_height={IMAGE_HEIGHT},
    samples={SAMPLES},
    engine="{RENDER_ENGINE}",
    max_points={MAX_POINTS}
)

# 执行渲染
renderer.process(
    use_batch_rendering={USE_BATCH_RENDERING},
    single_batch_id={SINGLE_BATCH_ID},
    max_batches={MAX_BATCHES}
)
'''
    
    # 保存渲染脚本
    temp_script_path = "/content/render_script.py"
    with open(temp_script_path, 'w') as f:
        f.write(render_script)
    
    print(f"\n开始渲染...")
    print(f"Blender路径: /content/blender/blender")
    print(f"脚本路径: {temp_script_path}")
    print("\n这可能需要几分钟时间，请耐心等待...")
    
    # 执行Blender渲染（headless模式）
    !/content/blender/blender --background --python {temp_script_path} 2>&1


In [ ]:
# 列出输出文件
import glob

output_files = glob.glob(f"{OUTPUT_FOLDER}/*.png")
if output_files:
    print(f"找到 {len(output_files)} 个渲染结果：")
    for f in sorted(output_files):
        file_size = os.path.getsize(f) / (1024 * 1024)  # MB
        print(f"  {os.path.basename(f)} ({file_size:.2f} MB)")
else:
    print("未找到渲染结果文件")
    print(f"请检查输出文件夹: {OUTPUT_FOLDER}")


In [ ]:
# 显示渲染结果（如果文件存在）
from IPython.display import Image, display
import glob

output_files = sorted(glob.glob(f"{OUTPUT_FOLDER}/*.png"))
if output_files:
    latest_file = output_files[-1]
    print(f"显示最新渲染结果: {os.path.basename(latest_file)}")
    display(Image(latest_file))
else:
    print("未找到渲染结果")


## 6. 下载渲染结果


In [ ]:
# 下载所有渲染结果
from google.colab import files
import zipfile

# 创建ZIP文件
zip_path = "/content/rendered_images.zip"
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for png_file in glob.glob(f"{OUTPUT_FOLDER}/*.png"):
        zipf.write(png_file, os.path.basename(png_file))

print(f"已创建ZIP文件: {zip_path}")
print("\n正在下载...")
files.download(zip_path)


## 7. 批量渲染多个PLY文件（可选）


In [ ]:
# 批量渲染配置
REPO_DIR = "/content/PointCloud_Render"
INPUT_FOLDER = "/content/input_ply"
OUTPUT_FOLDER = "/content/output_images"
PATTERN = "frame_*.ply"  # 文件匹配模式

# 渲染参数（与上面相同）
IMAGE_WIDTH = 1920
IMAGE_HEIGHT = 1080
RENDER_ENGINE = "cycles"
SAMPLES = 64
USE_BATCH_RENDERING = False
MAX_BATCHES = None
SINGLE_BATCH_ID = None
MAX_POINTS = None

# 查找所有PLY文件
import glob
ply_files = sorted(glob.glob(os.path.join(INPUT_FOLDER, PATTERN)))

if not ply_files:
    print(f"未找到匹配 {PATTERN} 的文件")
else:
    print(f"找到 {len(ply_files)} 个PLY文件")
    
    for idx, ply_file in enumerate(ply_files, 1):
        basename = os.path.basename(ply_file)
        print(f"\n[{idx}/{len(ply_files)}] 渲染: {basename}")
        
        # 创建渲染脚本
        render_script = f'''
import sys
sys.path.insert(0, '{REPO_DIR}')

from blender_point_cloud_renderer import BlenderPointCloudRenderer

renderer = BlenderPointCloudRenderer(
    file_path="{ply_file}",
    output_folder="{OUTPUT_FOLDER}",
    image_width={IMAGE_WIDTH},
    image_height={IMAGE_HEIGHT},
    samples={SAMPLES},
    engine="{RENDER_ENGINE}",
    max_points={MAX_POINTS}
)

renderer.process(
    use_batch_rendering={USE_BATCH_RENDERING},
    single_batch_id={SINGLE_BATCH_ID},
    max_batches={MAX_BATCHES}
)
'''
        
        script_path = f"/content/render_script_{idx}.py"
        with open(script_path, 'w') as f:
            f.write(render_script)
        
        # 执行渲染
        !/content/blender/blender --background --python {script_path} 2>&1 | tail -20
        
        # 清理脚本
        os.remove(script_path)
    
    print("\n批量渲染完成！")
